## <font color='cornflowerblue'> Dependencies

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

## <font color='cornflowerblue'> Functions

In [2]:
def get_financials(ticker:str) -> pd.DataFrame:
    
    bs = yf.Ticker(ticker).balance_sheet.iloc[:, :].sort_index(axis=1)
    ist = yf.Ticker(ticker).income_stmt.iloc[:, :].sort_index(axis=1)
    cf = yf.Ticker(ticker).cash_flow.iloc[:, :].sort_index(axis=1)

    if bs.iloc[:, 0].isna().sum() > 10:
        bs = bs.iloc[:, 1:]
    if ist.iloc[:, 0].isna().sum() > 10:
        ist = ist.iloc[:, 1:]
    if cf.iloc[:, 0].isna().sum() > 10:
        cf = cf.iloc[:, 1:]

    return bs, ist, cf

## <font color='cornflowerblue'> Stock

In [3]:
ticker = 'VRT'
bs, ist, cf = get_financials(ticker)

## <font color='cornflowerblue'> Financial Variables

In [4]:
# Extract key variables from the financial statements
revenue = ist.loc['Total Revenue']
cogs = ist.loc['Cost Of Revenue']
operating_expense = ist.loc['Operating Expense']
d_and_a = cf.loc['Depreciation And Amortization']
capex = cf.loc['Capital Expenditure']
change_in_wc = cf.loc['Change In Working Capital']

interest_expense = ist.loc['Net Non Operating Interest Income Expense'].iloc[-1]
#net_debt = bs.loc['Net Debt'].iloc[-1]
net_debt = bs.loc['Total Debt'].iloc[-1] - bs.loc['Cash Cash Equivalents And Short Term Investments'].iloc[-1]
total_debt = bs.loc['Total Debt'].iloc[-1]
n_shares = bs.loc['Ordinary Shares Number'].iloc[-1]

# Economic assumptions
perpetual_growth_rate = 0.03
risk_free_rate = 0.04

# Set up % to the set up the DCF model assuptions
growth_rate = revenue.pct_change().dropna()
cogs_pct_revenue = cogs / revenue
op_expenses_pct_revenue = operating_expense / revenue
capex_pct_revenue = -capex / revenue
d_and_a_pct_revenue = d_and_a / revenue
change_in_wc_pct_revenue = -change_in_wc / revenue
tax_rate = ist.loc['Tax Rate For Calcs'].mean()

/var/folders/60/rl4yk8jj3453bx7hbt074rbc0000gn/T/ipykernel_39390/3308952487.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  growth_rate = revenue.pct_change().dropna()


In [5]:
# Showing percentages to set up the DCF model assumptions
print('Growth Rate:', growth_rate.values.round(4))
print(f'COGS % Revenue: {cogs_pct_revenue.astype(float).values.round(4)}')
print(f'Operating Expenses % Revenue: {op_expenses_pct_revenue.astype(float).values.round(4)}')
print(f'CapEx % Revenue: {capex_pct_revenue.astype(float).values.round(4)}')
print(f'D&A % Revenue: {d_and_a_pct_revenue.astype(float).values.round(4)}')
print(f'Change in WC % Revenue: {change_in_wc_pct_revenue.astype(float).values.round(4)}')

Growth Rate: [0.2059 0.1674 0.2769]
COGS % Revenue: [0.7161 0.6502 0.6338 0.6368]
Operating Expenses % Revenue: [0.2439 0.2162 0.1937 0.1778]
CapEx % Revenue: [0.0195 0.0196 0.023  0.0221]
D&A % Revenue: [0.0531 0.0395 0.0346 0.0302]
Change in WC % Revenue: [ 0.0789 -0.0097 -0.0142 -0.0332]


## <font color='cornflowerblue'> Assumptions

In [6]:
#growth_forecast = [0.11, 0.0995, 0.10, 0.1050, 0.11]

#cogs_forecast = [0.1950, 0.1940, 0.1940, 0.1920, 0.1930]

#opex_forecast = [0.14, 0.1395, 0.1380, 0.1350, 0.1350]

growth_forecast = [growth_rate.mean()] * 5

cogs_forecast = [cogs_pct_revenue.mean()] * 5

opex_forecast = [op_expenses_pct_revenue.mean()] * 5

capex_forecast = [capex_pct_revenue.mean()] * 5

da_forecast = [d_and_a_pct_revenue.mean()] * 5

wc_forecast = [change_in_wc_pct_revenue.mean()] * 5

## <font color='cornflowerblue'> Projections

In [7]:
last_rev = revenue.iloc[-1]

projections = []

# Projecting the next 5 years of financials based on the assumptions
for i in range(5):
    last_rev = last_rev * (1 + growth_forecast[i])
    proj_cogs = last_rev * cogs_forecast[i]
    proj_opex = last_rev * opex_forecast[i]
    proj_capex = last_rev * capex_forecast[i]
    proj_da = last_rev * da_forecast[i]
    proj_wc = last_rev * wc_forecast[i]
    ebit = last_rev - proj_cogs - proj_opex
    fcff = ebit * (1 - tax_rate) + proj_da - proj_capex - proj_wc
    projections.append({
        'Revenue': last_rev,
        'COGS': proj_cogs,
        'Operating Expense': proj_opex,
        'EBIT': ebit,
        'CapEx': proj_capex,
        'D&A': proj_da,
        'Change in WC': proj_wc,
        'FCFF': fcff
    })

projections_df = pd.DataFrame(projections)
projections_df.index = [f'Year {i+1}' for i in range(5)]
projections_df

,Revenue,COGS,Operating Expense,EBIT,CapEx,D&A,Change in WC,FCFF
Year 1,1.244665e+10,8.205081e+09,2.587840e+09,1.653726e+09,2.620779e+08,4.896458e+08,6.782564e+07,1.426910e+09
Year 2,1.514375e+10,9.983065e+09,3.148606e+09,2.012077e+09,3.188684e+08,5.957486e+08,8.252298e+07,1.736111e+09
Year 3,1.842529e+10,1.214632e+10,3.830887e+09,2.448080e+09,3.879649e+08,7.248431e+08,1.004051e+08,2.112314e+09
Year 4,2.241792e+10,1.477835e+10,4.661014e+09,2.978562e+09,4.720341e+08,8.819115e+08,1.221622e+08,2.570038e+09
Year 5,2.727573e+10,1.798071e+10,5.671023e+09,3.623995e+09,5.743206e+08,1.073016e+09,1.486339e+08,3.126947e+09


## <font color='cornflowerblue'> Valuation

### <font color='skyblue'> WACC Computation

In [8]:
# Historical data for the stock and the market (VOO)
data = yf.download([ticker, 'VOO'], period='5y', interval='1d', progress=False, auto_adjust=True)['Close']
rt = data.pct_change().dropna()

# Market Cap
market_cap = data[ticker].iloc[-2] * n_shares
total_capitalization = market_cap + total_debt

# Weights for WACC
weight_equity = market_cap / total_capitalization
weight_debt = total_debt / total_capitalization

# Compute beta
cov_matrix = rt.cov()
beta = cov_matrix.iloc[0,1] / cov_matrix.iloc[1,1]

# Get market return
market_return = np.mean(rt['VOO']) * 252

# Use CAPM model to calculate cost of equity
cost_of_equity = risk_free_rate + beta * (market_return - risk_free_rate)

# Estimate cost of debt

if interest_expense < 0:
    interest_expense = -interest_expense
    
cost_of_debt = interest_expense / bs.loc['Total Debt'].iloc[-2:].mean()
at_cost_of_debt = cost_of_debt * (1 - tax_rate)

# Compute WACC
wacc = weight_equity * cost_of_equity + weight_debt * at_cost_of_debt

print(f'Cost of Equity: {cost_of_equity:.4f}')
print(f'Cost of Debt: {cost_of_debt:.4f}')
print(f'WACC: {wacc:.4f}')

Cost of Equity: 0.0548
Cost of Debt: 0.0270
WACC: 0.0540


In [9]:
# Compute terminal value using the Gordon Growth Model
terminal_value = (projections_df['FCFF'].iloc[-1] * (1 + perpetual_growth_rate)) / (wacc - perpetual_growth_rate)

# Add terminal value to the last year's FCFF
full_fcff = projections_df['FCFF'].copy()
full_fcff.iloc[-1] += terminal_value

# Discount FCFF to present value
present_value_fcff = full_fcff / (1 + wacc) ** np.arange(1, 6)

# Get enterprise value by summing the present value of FCFF
enterprise_value = present_value_fcff.sum()

# Compute equity value
equity_value = enterprise_value - net_debt

# Intrinsic value per share
intrinsic_value_per_share = equity_value / n_shares

print(f'Intrinsic Value per Share: ${intrinsic_value_per_share:.2f}')
print(f'Market Price per Share: ${data[ticker].iloc[-2]:.2f}')
print(f'Upside/Downside Potential: {((intrinsic_value_per_share / data[ticker].iloc[-2]) - 1):.4%}')

Intrinsic Value per Share: $290.70
Market Price per Share: $325.57
Upside/Downside Potential: -10.7110%
